In [ ]:
# JupyterLite 浏览器 ONNX Runtime：兼容原考试 onnxruntime API，底层使用 onnxruntime-web
import sys
import types
from types import SimpleNamespace

import numpy as _np
from js import Object
from pyodide.code import run_js
from pyodide.ffi import can_run_sync, run_sync, to_js

_browser_ort_bridge = run_js(r"""
(() => {
  const origin = self.location.origin;
  const sessions = new Map();
  let manifestPromise;
  let runtimePromise;

  const digest = async buffer => {
    const hash = await crypto.subtle.digest("SHA-256", buffer);
    return [...new Uint8Array(hash)].map(value => value.toString(16).padStart(2, "0")).join("");
  };

  const loadManifest = async () => {
    manifestPromise ??= fetch(`${origin}/labs/manifest.json`).then(response => {
      if (!response.ok) throw new Error("模型清单加载失败");
      return response.json();
    });
    return manifestPromise;
  };

  const loadRuntime = async () => {
    if (!runtimePromise) {
      const runtimeBase = `${origin}/labs/runtime/onnxruntime-web/1.27.0/`;
      runtimePromise = import(`${runtimeBase}ort.wasm.bundle.min.js`).then(ort => {
        ort.env.wasm.numThreads = 1;
        ort.env.wasm.wasmPaths = runtimeBase;
        return ort;
      });
    }
    return runtimePromise;
  };

  const loadModelBytes = async model => {
    const cache = await caches.open("ai-trainer-onnx-v2");
    const cacheKey = new Request(`${origin}/labs/model-cache/${model.modelId}.onnx`);
    const cached = await cache.match(cacheKey);
    if (cached) {
      const cachedBuffer = await cached.arrayBuffer();
      if (await digest(cachedBuffer) === model.sha256) return new Uint8Array(cachedBuffer);
    }

    const chunks = [];
    let total = 0;
    for (const part of model.chunks) {
      const response = await fetch(`${origin}${part.url}`);
      if (!response.ok) throw new Error(`模型分片加载失败：${part.url}`);
      const buffer = await response.arrayBuffer();
      if (await digest(buffer) !== part.sha256) throw new Error(`模型分片校验失败：${part.url}`);
      const bytes = new Uint8Array(buffer);
      chunks.push(bytes);
      total += bytes.byteLength;
    }

    const modelBytes = new Uint8Array(total);
    let offset = 0;
    for (const chunk of chunks) {
      modelBytes.set(chunk, offset);
      offset += chunk.byteLength;
    }
    if (await digest(modelBytes.buffer) !== model.sha256) throw new Error("模型完整性校验失败");
    await cache.put(cacheKey, new Response(modelBytes, { headers: { "content-type": "application/octet-stream" } }));
    return modelBytes;
  };

  const metadata = (names, items) => names.map((name, index) => {
    const item = items[index];
    return {
      name,
      type: String(item?.type ?? "float32"),
      shape: [...(item?.shape ?? [])],
    };
  });

  const describe = record => ({
    id: record.id,
    model: record.model.name,
    runtime: record.model.browserRuntime,
    variant: record.model.sourceVariant,
    inputs: metadata(record.session.inputNames, record.session.inputMetadata),
    outputs: metadata(record.session.outputNames, record.session.outputMetadata),
  });

  const create = async taskCode => {
    const existing = sessions.get(taskCode);
    if (existing) return describe(existing);
    const manifest = await loadManifest();
    const model = manifest.tasks?.[taskCode]?.model;
    if (!model) throw new Error(`任务 ${taskCode} 没有浏览器模型`);
    const [ort, modelBytes] = await Promise.all([loadRuntime(), loadModelBytes(model)]);
    const session = await ort.InferenceSession.create(modelBytes, { executionProviders: ["wasm"] });
    const record = { id: taskCode, model, ort, session };
    sessions.set(taskCode, record);
    return describe(record);
  };

  const typedData = (type, values) => {
    const data = Array.from(values);
    if (type === "float64") return new Float64Array(data);
    if (type === "int8") return new Int8Array(data);
    if (type === "uint8" || type === "bool") return new Uint8Array(data);
    if (type === "int16") return new Int16Array(data);
    if (type === "uint16") return new Uint16Array(data);
    if (type === "int32") return new Int32Array(data);
    if (type === "uint32") return new Uint32Array(data);
    if (type === "int64") return BigInt64Array.from(data, value => BigInt(value));
    if (type === "uint64") return BigUint64Array.from(data, value => BigInt(value));
    return new Float32Array(data);
  };

  const serializeTensor = (name, tensor) => ({
    name,
    type: tensor.type,
    dims: [...tensor.dims],
    data: Array.from(tensor.data, value => typeof value === "bigint" ? value.toString() : value),
  });

  const run = async (sessionId, inputItems, requestedOutputs) => {
    const record = sessions.get(sessionId);
    if (!record) throw new Error("ONNX 会话不存在，请重新加载实验环境");
    const feeds = {};
    for (const item of Array.from(inputItems)) {
      feeds[item.name] = new record.ort.Tensor(
        item.type,
        typedData(item.type, item.data),
        Array.from(item.dims, Number),
      );
    }
    const outputMap = await record.session.run(feeds);
    const names = requestedOutputs?.length ? Array.from(requestedOutputs) : record.session.outputNames;
    return names.map(name => serializeTensor(name, outputMap[name]));
  };

  const smoke = async taskCode => {
    const info = await create(taskCode);
    const record = sessions.get(taskCode);
    const feeds = {};
    const inputs = [];
    for (const item of info.inputs) {
      const shape = (item.shape.length ? item.shape : [1]).map(value => Number.isInteger(value) && value > 0 ? value : 1);
      const size = shape.reduce((left, right) => left * right, 1);
      feeds[item.name] = new record.ort.Tensor(item.type, typedData(item.type, new Array(size).fill(0)), shape);
      inputs.push({ name: item.name, type: item.type, shape });
    }
    const outputMap = await record.session.run(feeds);
    const outputs = record.session.outputNames.map(name => {
      const tensor = outputMap[name];
      return { name, type: tensor.type, shape: [...tensor.dims] };
    });
    return { model: info.model, runtime: info.runtime, variant: info.variant, inputs, outputs };
  };

  return { create, run, smoke };
})()
""")


def _browser_value(value):
    return value.to_py() if hasattr(value, "to_py") else value


def _numpy_dtype(onnx_type):
    return {
        "float32": _np.float32,
        "float64": _np.float64,
        "int8": _np.int8,
        "uint8": _np.uint8,
        "bool": _np.bool_,
        "int16": _np.int16,
        "uint16": _np.uint16,
        "int32": _np.int32,
        "uint32": _np.uint32,
        "int64": _np.int64,
        "uint64": _np.uint64,
    }.get(onnx_type, _np.float32)


class InferenceSession:
    """浏览器版 onnxruntime.InferenceSession 兼容层。"""

    def __init__(self, model_path, providers=None, **_options):
        if not can_run_sync():
            raise RuntimeError("当前浏览器不支持同步 ONNX 桥接，请升级到最新版 Chromium 浏览器")
        info = _browser_value(run_sync(_browser_ort_bridge.create("3.2.5")))
        self._session_id = info["id"]
        self._model_path = str(model_path)
        self._providers = list(providers or ["WasmExecutionProvider"])
        self._inputs = [SimpleNamespace(**item) for item in info["inputs"]]
        self._outputs = [SimpleNamespace(**item) for item in info["outputs"]]

    def get_inputs(self):
        return self._inputs

    def get_outputs(self):
        return self._outputs

    def get_providers(self):
        return self._providers

    def run(self, output_names, input_feed, run_options=None):
        del run_options
        inputs = []
        for name, value in input_feed.items():
            array = _np.ascontiguousarray(value)
            onnx_type = {
                "f": "float32" if array.dtype.itemsize <= 4 else "float64",
                "i": f"int{array.dtype.itemsize * 8}",
                "u": f"uint{array.dtype.itemsize * 8}",
                "b": "bool",
            }.get(array.dtype.kind, "float32")
            inputs.append({
                "name": str(name),
                "type": onnx_type,
                "dims": list(array.shape),
                "data": array.reshape(-1).tolist(),
            })
        requested = [] if output_names is None else list(output_names)
        result = run_sync(_browser_ort_bridge.run(
            self._session_id,
            to_js(inputs, dict_converter=Object.fromEntries),
            to_js(requested),
        ))
        outputs = _browser_value(result)
        return [
            _np.asarray(item["data"], dtype=_numpy_dtype(item["type"])).reshape(tuple(item["dims"]))
            for item in outputs
        ]


_onnxruntime = types.ModuleType("onnxruntime")
_onnxruntime.InferenceSession = InferenceSession
_onnxruntime.__version__ = "1.27.0-web"
sys.modules["onnxruntime"] = _onnxruntime


async def browser_onnx_check(task_code="3.2.5"):
    """执行一次真实的浏览器 WASM 推理，并返回张量结构。"""
    result = await _browser_ort_bridge.smoke(task_code)
    return _browser_value(result)


onnx_result = await browser_onnx_check()
onnx_result


In [ ]:
import os
import time
import cv2
import numpy as np
import vision.utils.box_utils_numpy as box_utils
import onnxruntime as ort

# 定义预测函数，对模型输出的边界框和置信度进行后处理
def predict(width, height, confidences, boxes, prob_threshold, iou_threshold=0.3, top_k=-1):
    boxes = boxes[0]
    confidences = confidences[0]
    picked_box_probs = []
    picked_labels = []
    for class_index in range(1, confidences.shape[1]):
        probs = confidences[:, class_index]
        mask = probs > prob_threshold
        probs = probs[mask]
        if probs.shape[0] == 0:
            continue
        subset_boxes = boxes[mask, :]
        box_probs = np.concatenate([subset_boxes, probs.reshape(-1, 1)], axis=1)
        box_probs = box_utils.hard_nms(box_probs,
                                       iou_threshold=iou_threshold,
                                       top_k=top_k,
                                       )
        picked_box_probs.append(box_probs)
        picked_labels.extend([class_index] * box_probs.shape[0])
    if not picked_box_probs:
        return np.array([]), np.array([]), np.array([])
    picked_box_probs = np.concatenate(picked_box_probs)
    picked_box_probs[:, 0] *= width
    picked_box_probs[:, 1] *= height
    picked_box_probs[:, 2] *= width
    picked_box_probs[:, 3] *= height
    return picked_box_probs[:, :4].astype(np.int32), np.array(picked_labels), picked_box_probs[:, 4]

# 从标签文件中读取每一行，并去除行首尾的空白字符，得到类别名称列表 2分
class_names = [_______________ for name in open('voc-model-labels.txt').readlines()]

# 创建 ONNX Runtime 的推理会话，用于运行模型进行推理 2分
ort_session = _______________('version-RFB-320.onnx')

# 获取模型输入的名称 2分
input_name = _______________()[0].name

# 定义保存检测结果图像的目录路径
result_path = "./detect_imgs_results_onnx"

# 定义置信度阈值，用于筛选出置信度较高的检测结果
threshold = 0.7
# 定义存储待检测图像的目录路径
path = "imgs"
# 用于统计所有图像中检测到的目标框总数，初始化为 0
sum = 0

# 如果保存结果的目录不存在，则创建该目录 2分
if not os.path.exists(result_path):
    os._______________
    
# 获取指定目录下的所有文件和文件夹名称列表
listdir = os.listdir(path)

# 遍历目录下的每个文件
for file_path in listdir:
    # 拼接图像文件的完整路径
    img_path = os.path.join(path, file_path)
    # 使用 OpenCV 读取图像文件 2分
    orig_image = _______________
    # 将图像从 BGR 颜色空间转换为 RGB 颜色空间（许多模型要求输入为 RGB 格式）
    image = cv2.cvtColor(orig_image, cv2.COLOR_BGR2RGB)
    # 将图像调整为 320x240 的尺寸（符合模型输入的尺寸要求） 2分
    image = _______________(_______________, (320, 240))
    # 定义图像归一化的均值数组 2分
    image_mean = _______________([127, 127, 127])
    # 对图像进行归一化处理，减去均值并除以 128
    image = (image - image_mean) / 128
    # 将图像的维度从 (高度, 宽度, 通道数) 转换为 (通道数, 高度, 宽度)
    image = np.transpose(image, [2, 0, 1])
    # 在第一个维度上扩展一个维度，将图像变为 (1, 通道数, 高度, 宽度)，以符合模型输入的维度要求  1分
    image = _______________(image, axis=0)
    # 将图像数据类型转换为 float32 类型
    image = image.astype(np.float32)
    # 记录开始时间，用于计算模型推理的耗时
    time_time = time.time()
    # 使用 ONNX Runtime 运行模型，输入图像数据，得到模型输出的置信度和边界框  2分
    confidences, boxes = _______________(None, {input_name: image})
    # 计算并打印模型推理的耗时
    print("cost time:{}".format(time.time() - time_time))
    # 调用 predict 函数对模型输出的边界框和置信度进行后处理，得到最终的边界框、类别标签和置信度
    boxes, labels, probs = predict(orig_image.shape[1], orig_image.shape[0], confidences, boxes, threshold)
    # 遍历每个检测到的目标框
    for i in range(boxes.shape[0]):
        # 获取当前目标框的坐标
        box = boxes[i, :]
        # 生成当前目标框的标签字符串，包含类别名称和置信度
        label = f"{class_names[labels[i]]}: {probs[i]:.2f}"

        # 在原始图像上绘制目标框，颜色为 (255, 255, 0)，线条粗细为 4
        cv2.rectangle(orig_image, (box[0], box[1]), (box[2], box[3]), (255, 255, 0), 4)
        # 将绘制了目标框的图像保存到结果目录中
        cv2.imwrite(os.path.join(result_path, file_path), orig_image)
    # 累加当前图像中检测到的目标框数量到总数中
    sum += boxes.shape[0]
# 打印所有图像中检测到的目标框总数
print("sum:{}".format(sum))